# Quadratic Regression — foF2 Prediction

**Features:** DayOfYear, Hour, Longitude, Latitude, F10.7  
**Target:** foF2 (MHz)  
**Pipeline:** PolynomialFeatures(degree=2) → StandardScaler → LinearRegression  
**Split:** 70 % train / 15 % val / 15 % test (random_state=42)  
**Outputs:** `outputs/quadratic_model.joblib`, `outputs/scatter_quadratic.png`, `outputs/predictions_quadratic.csv`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

## Paths & Configuration

In [ ]:
import os

_cwd = Path(os.getcwd())
if (_cwd / 'FINAL_MASTER.csv').exists():
    project_root = _cwd
elif (_cwd.parent.parent / 'FINAL_MASTER.csv').exists():
    project_root = _cwd.parent.parent
else:
    raise FileNotFoundError('Cannot find FINAL_MASTER.csv — run Jupyter from the project root or models/notebooks/')

data_path  = project_root / 'FINAL_MASTER.csv'
output_dir = project_root / 'outputs'
output_dir.mkdir(exist_ok=True)

RANDOM_STATE = 42
FEATURES     = ['DayOfYear', 'Hour', 'Longitude', 'Latitude', 'F10.7']
TARGET       = 'foF2'
POLY_DEGREE  = 2

print(f'Project root : {project_root}')
print(f'Data         : {data_path}')
print(f'Poly degree  : {POLY_DEGREE}')

## Load Data

In [ ]:
df = pd.read_csv(data_path)
print(f'Shape   : {df.shape}')
print(f'Stations: {sorted(df["Station"].unique())}')
df.head()

In [ ]:
print('Null values:')
print(df.isnull().sum())

## Train / Val / Test Split  (70 / 15 / 15)

In [ ]:
X        = df[FEATURES].values
y        = df[TARGET].values
stations = df['Station'].values

X_train, X_temp, y_train, y_temp, s_train, s_temp = train_test_split(
    X, y, stations, test_size=0.30, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test, s_val, s_test = train_test_split(
    X_temp, y_temp, s_temp, test_size=0.50, random_state=RANDOM_STATE)

print(f'Train : {len(X_train):,}  ({len(X_train)/len(X)*100:.1f} %)')
print(f'Val   : {len(X_val):,}  ({len(X_val)/len(X)*100:.1f} %)')
print(f'Test  : {len(X_test):,}  ({len(X_test)/len(X)*100:.1f} %)')

## Build Pipeline & Train

In [ ]:
model = Pipeline([
    ('poly',      PolynomialFeatures(degree=POLY_DEGREE, include_bias=False)),
    ('scaler',    StandardScaler()),
    ('regressor', LinearRegression()),
])
model.fit(X_train, y_train)
print(f'Training complete.  Pipeline: {model}')

## Metrics Helper

In [ ]:
def metrics(y_true, y_pred):
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    yt, yp = y_true[mask], y_pred[mask]
    if len(yt) == 0:
        return None
    return dict(
        MSE  = float(mean_squared_error(yt, yp)),
        RMSE = float(np.sqrt(mean_squared_error(yt, yp))),
        MAE  = float(mean_absolute_error(yt, yp)),
        R2   = float(r2_score(yt, yp)),
        Bias = float(np.mean(yp - yt)),
        N    = int(mask.sum()),
    )

## Test-Set Metrics (Diagnostic)

In [ ]:
m = metrics(y_test, model.predict(X_test))
print(f"MSE={m['MSE']:.4f}  RMSE={m['RMSE']:.4f}  MAE={m['MAE']:.4f}  R2={m['R2']:.4f}  Bias={m['Bias']:+.4f}  N={m['N']:,}")

## All-Data Metrics + Per-Station Breakdown

In [ ]:
y_pred_all = model.predict(X)
m_all = metrics(y, y_pred_all)

print('ALL DATA')
print(f"  MSE={m_all['MSE']:.4f}  RMSE={m_all['RMSE']:.4f}  MAE={m_all['MAE']:.4f}  R2={m_all['R2']:.4f}  N={m_all['N']:,}")
print()
print('PER STATION')
for s in sorted(np.unique(stations)):
    mask = stations == s
    ms = metrics(y[mask], y_pred_all[mask])
    print(f"  {s:<14} RMSE={ms['RMSE']:.4f}  R2={ms['R2']:.4f}  Bias={ms['Bias']:+.4f}  N={ms['N']}")

## Scatter Plot (All Data)

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y, y_pred_all, alpha=0.3, s=8)
lims = [min(y.min(), y_pred_all.min()) - 0.3, max(y.max(), y_pred_all.max()) + 0.3]
plt.plot(lims, lims, 'r--', lw=1.5, label='1:1')
plt.xlim(lims); plt.ylim(lims)
plt.xlabel('Observed foF2 (MHz)'); plt.ylabel('Predicted foF2 (MHz)')
plt.title(f"Quadratic Regression (deg={POLY_DEGREE}) — All Data\nR²={m_all['R2']:.4f}  RMSE={m_all['RMSE']:.4f}")
plt.legend(); plt.tight_layout()
plt.savefig(output_dir / 'scatter_quadratic.png', dpi=150)
plt.show()
print('Saved: outputs/scatter_quadratic.png')

## Save Model & Predictions

In [ ]:
joblib.dump(model, output_dir / 'quadratic_model.joblib')
print('Model saved: outputs/quadratic_model.joblib')

pd.DataFrame({
    'Station':   stations,
    'DayOfYear': df['DayOfYear'].values,
    'Hour':      df['Hour'].values,
    'foF2_obs':  y,
    'foF2_pred': y_pred_all,
}).to_csv(output_dir / 'predictions_quadratic.csv', index=False)
print(f'Predictions saved: outputs/predictions_quadratic.csv  ({len(df):,} rows)')